# Fine-Tuning BERT for POS Tagging & Chunking
**NLP Assignment 5 — Token Classification**

---
### Pipeline Overview
```
Raw Data → Tokenization → Label Alignment → Model Training → Evaluation → Inference → Comparison
```





## Task 1: Dataset Selection

Using **CoNLL-2003** dataset from HuggingFace — it contains both **POS tags** and **Chunk tags** in a single dataset, making it ideal for this assignment.

**Label Types:**
- **POS Tags**: NN, VBZ, NNP, IN, JJ, etc. (grammar-level)
- **Chunk Tags**: B-NP, I-NP, B-VP, B-PP, O (phrase-level, IOB format)


In [12]:
# Install required libraries (run once in Colab)
!pip install transformers datasets==2.10.0 seqeval evaluate --quiet --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.0/469.0 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.6/202.6 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
# import
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
import evaluate
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {'cuda' if torch.cuda.is_available() else 'cpu'}")

PyTorch version : 2.10.0+cu128
Device          : cuda


In [16]:
!pip install datasets==2.14.0 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 492.2/492.2 kB 9.3 MB/s eta 0:00:00


In [22]:
!pip install -q datasets --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.5/202.5 kB 22.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.


In [1]:
from datasets import load_dataset

dataset = load_dataset("wikiann", "en")
print(dataset)
print(dataset["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

en/validation-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

en/test-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

en/train-00000-of-00001.parquet:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
})
{'tokens': ['R.H.', 'Saunders', '(', 'St.', 'Lawrence', 'River', ')', '(', '968', 'MW', ')'], 'ner_tags': [3, 4, 0, 3, 4, 4, 0, 0, 0, 0, 0], 'langs': ['en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en'], 'spans': ['ORG: R.H. Saunders', 'ORG: St. Lawrence River']}


In [2]:
print("Dataset structure:")
print(dataset)

print("\nOne sample:")
print(dataset["train"][0])

print("\nLabel names:")
label_names = dataset["train"].features["ner_tags"].feature.names
print(label_names)
print(f"Total labels: {len(label_names)}")

Dataset structure:
DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
})

One sample:
{'tokens': ['R.H.', 'Saunders', '(', 'St.', 'Lawrence', 'River', ')', '(', '968', 'MW', ')'], 'ner_tags': [3, 4, 0, 3, 4, 4, 0, 0, 0, 0, 0], 'langs': ['en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en'], 'spans': ['ORG: R.H. Saunders', 'ORG: St. Lawrence River']}

Label names:
['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']
Total labels: 7


In [5]:
from transformers import AutoTokenizer # Ensure AutoTokenizer is imported

#  Load Tokenizer
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer loaded!")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded!


In [6]:
#  Label Alignment Function
label_names = dataset["train"].features["ner_tags"].feature.names

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=MAX_LEN,
        is_split_into_words=True,
    )

    all_labels = []
    for i, label_seq in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        aligned = []
        previous_wid = None

        for wid in word_ids:
            if wid is None:
                # [CLS] and [SEP] → ignore
                aligned.append(-100)
            elif wid != previous_wid:
                # First subword → real label
                aligned.append(label_seq[wid])
            else:
                # Remaining subwords → ignore
                aligned.append(-100)
            previous_wid = wid

        all_labels.append(aligned)

    tokenized["labels"] = all_labels
    return tokenized

In [7]:
# Apply Tokenization
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names
)
print(tokenized_dataset)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

DatasetDict({
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 20000
    })
})


In [8]:
# Verify One Sample
sample = dataset["train"][0]
print("Original tokens:", sample["tokens"][:5])
print("Original labels:", [label_names[t] for t in sample["ner_tags"][:5]])

enc = tokenizer(sample["tokens"], is_split_into_words=True)
print("Subword tokens :", tokenizer.convert_ids_to_tokens(enc["input_ids"][:8]))
print("Word IDs       :", enc.word_ids()[:8])

Original tokens: ['R.H.', 'Saunders', '(', 'St.', 'Lawrence']
Original labels: ['B-ORG', 'I-ORG', 'O', 'B-ORG', 'I-ORG']
Subword tokens : ['[CLS]', 'r', '.', 'h', '.', 'saunders', '(', 'st']
Word IDs       : [None, 0, 0, 0, 0, 1, 2, 3]


In [9]:
#  Build Model
id2label = {i: l for i, l in enumerate(label_names)}
label2id = {l: i for i, l in id2label.items()}

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)
print(f"Model loaded | Number of labels: {len(label_names)}")
print(f"Labels: {label_names}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded | Number of labels: 7
Labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


In [10]:
#Metrics Function
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    true_preds = []
    true_labels = []

    for pred_seq, label_seq in zip(predictions, labels):
        token_preds = []
        token_labels = []
        for p, l in zip(pred_seq, label_seq):
            if l != -100:
                token_preds.append(label_names[p])
                token_labels.append(label_names[l])
        true_preds.append(token_preds)
        true_labels.append(token_labels)

    result = seqeval.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": result["overall_precision"],
        "recall"   : result["overall_recall"],
        "f1"       : result["overall_f1"],
        "accuracy" : result["overall_accuracy"],
    }

In [12]:
# Data Collator & Training Args
data_collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir="./ner_model",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch", # Changed from evaluation_strategy to eval_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
    report_to="none",
)
print("Training args set!")

Training args set!


In [13]:
# Train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer, # Pass tokenizer to Trainer for proper handling of padding/truncation
)

trainer.train()

/tmp/ipykernel_4451/239271330.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.274500,0.252307,0.779151,0.812420,0.795438,0.922887
2,0.190800,0.243467,0.807626,0.830902,0.819099,0.929124
3,0.155100,0.249339,0.809116,0.835930,0.822304,0.930268


Training complete!


In [14]:
#   Evaluate
results = trainer.evaluate(tokenized_dataset["test"])

print("=== Test Set Results ===")
print(f"  Precision : {results['eval_precision']:.4f}")
print(f"  Recall    : {results['eval_recall']:.4f}")
print(f"  F1 Score  : {results['eval_f1']:.4f}")
print(f"  Accuracy  : {results['eval_accuracy']:.4f}")

=== Test Set Results ===
  Precision : 0.7999
  Recall    : 0.8275
  F1 Score  : 0.8135
  Accuracy  : 0.9278


In [19]:
#  Inference Function
def predict_tags(sentence):
    words = sentence.split()
    tokenized_inputs = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=128,
    )

    # Get word_ids before moving to device (as moving converts it to a dict)
    word_ids = tokenized_inputs.word_ids()

    # Move inputs to the same device as the model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    inputs = {k: v.to(device) for k, v in tokenized_inputs.items()}
    model.to(device) # Ensure model is on the correct device

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    predictions = torch.argmax(outputs.logits, dim=-1)[0].tolist()

    word_preds = {}
    for wid, pred in zip(word_ids, predictions):
        if wid is not None and wid not in word_preds:
            word_preds[wid] = id2label[pred]

    return words, [word_preds[i] for i in range(len(words))]

In [20]:
# /Run Inference
sentence = "John works at Google in California"

words, tags = predict_tags(sentence)

print("Input:", sentence)
print()
print(f"{'Word':<15} {'Predicted Tag'}")
print("-" * 30)
for word, tag in zip(words, tags):
    print(f"{word:<15} {tag}")

Input: John works at Google in California

Word            Predicted Tag
------------------------------
John            B-PER
works           O
at              O
Google          B-ORG
in              O
California      I-ORG


In [21]:
#  Comparison Table
import pandas as pd

comparison = pd.DataFrame({
    "Aspect"    : ["What it does", "Level", "Tag format", "Example", "Difficulty"],
    "POS Tagging" : [
        "Labels grammar role of each word",
        "Word level",
        "Single flat tag per word (NN, VBZ, IN)",
        "John=NNP, works=VBZ",
        "Easier"
    ],
    "Chunking / NER" : [
        "Groups words into named entities or phrases",
        "Phrase level",
        "IOB format (B-ORG, I-ORG, O)",
        "John=B-PER, Google=B-ORG",
        "Harder"
    ],
})
print(comparison.set_index("Aspect").to_string())

                                         POS Tagging                               Chunking / NER
Aspect                                                                                           
What it does        Labels grammar role of each word  Groups words into named entities or phrases
Level                                     Word level                                 Phrase level
Tag format    Single flat tag per word (NN, VBZ, IN)                 IOB format (B-ORG, I-ORG, O)
Example                          John=NNP, works=VBZ                     John=B-PER, Google=B-ORG
Difficulty                                    Easier                                       Harder


## Report

### What is Token Classification?
Token classification assigns a label to every single word (token) in a sentence.
This assignment uses WikiANN — a dataset where every word is labeled as a
Person (PER), Organization (ORG), Location (LOC), or Other (O).

### What is IOB Format?
B- means Beginning of an entity. I- means Inside. O means Outside.
Example: "New York" → B-LOC I-LOC

### Biggest Challenge — Label Alignment
BERT splits words into subwords. "California" might become ["California"] but
"Saunders" might become ["Sau", "##nders"]. Only the first piece gets the real
label. The rest get -100 so training ignores them.

### What I Observed
- The model learns entity boundaries (B- vs I-) well after 3 epochs
- O tags are most common so accuracy looks high even early on
- F1 score is the most honest metric here — it ignores the easy O tags